In [ ]:
!pip -q uninstall -y torch torchvision torchaudio
!pip -q cache purge

!pip -q install --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 108.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 112.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 64.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 88.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 834.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip -q install -U transformers

import torch, torchvision, torchaudio
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)

from transformers import pipeline
clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print(clf("This is fine."))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 19.2 MB/s eta 0:00:00


In [ ]:


import re
import tarfile
import urllib.request
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from collections import Counter

IMDB_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

def download_and_extract_imdb(data_dir: str = "/content/data") -> Path:
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    tgz_path = data_dir / "aclImdb_v1.tar.gz"
    extract_root = data_dir / "aclImdb"

    if not extract_root.exists():
        if not tgz_path.exists():
            print(f"Downloading to {tgz_path} ...")
            urllib.request.urlretrieve(IMDB_URL, tgz_path)
        print(f"Extracting to {data_dir} ...")
        with tarfile.open(tgz_path, "r:gz") as tf:
            tf.extractall(path=data_dir)

    return extract_root

_word_re = re.compile(r"[A-Za-z0-9']+")
def tokenize(text: str) -> List[str]:
    return _word_re.findall(text.lower())

BLOCKLIST = {
    "better", "best", "worse", "worst",

    "amazing", "awesome", "awful", "terrible", "excellent", "horrible",
    "fantastic", "wonderful", "perfect", "great", "bad", "good",
    "disappointing", "boring", "brilliant", "poor",

    "love", "hate", "liked", "disliked", "adore", "despise"
}

VALENCE_PATTERNS = [
    re.compile(r"\bbetter\b", re.IGNORECASE),
    re.compile(r"\bworse\b", re.IGNORECASE),
    re.compile(r"\bbest\b", re.IGNORECASE),
    re.compile(r"\bworst\b", re.IGNORECASE),
    re.compile(r"\bmore\s+\w+\s+than\b", re.IGNORECASE),
    re.compile(r"\bless\s+\w+\s+than\b", re.IGNORECASE),
]

def contains_blocklisted_word(prefix: str) -> Tuple[bool, Optional[str]]:
    toks = tokenize(prefix)
    for tok in toks:
        if tok in BLOCKLIST:
            return True, tok
    return False, None

def matches_valence_pattern(prefix: str) -> Tuple[bool, Optional[str]]:
    for pat in VALENCE_PATTERNS:
        if pat.search(prefix):
            return True, pat.pattern
    return False, None

def iter_review_texts(aclImdb_root: Path, split: str = "train", labels=("pos", "neg")):
    for label in labels:
        folder = aclImdb_root / split / label
        for fp in folder.glob("*.txt"):
            yield fp.read_text(encoding="utf-8", errors="ignore")


In [ ]:
# --- Sentiment scoring ---
sent = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def load_second_pass_classifier():
    candidate_models = [

        "siebert/sentiment-roberta-large-english",
        "distilbert-base-uncased-finetuned-sst-2-english",
    ]

    for model_name in candidate_models:
        try:
            print(f"Trying second-pass model: {model_name}")
            clf = pipeline(
                "sentiment-analysis",
                model=model_name,
                use_safetensors=True
            )
            print(f"Loaded second-pass model: {model_name}")
            return clf, model_name
        except Exception as e:
            print(f"Failed {model_name}: {e}")

    print("Warning: no second-pass classifier could be loaded. Falling back to single-pass filtering.")
    return None, None

sent2, sent2_name = load_second_pass_classifier()
print("sent2_name =", sent2_name)

def p_pos_from_result(res) -> float:

    if res["label"].upper().startswith("POS"):
        return float(res["score"])
    else:
        return 1.0 - float(res["score"])

LOW_NEUTRAL = 0.47
HIGH_NEUTRAL = 0.53

def p_pos_from_result_any(res: Dict) -> float:
    label = str(res["label"]).strip().upper()
    score = float(res["score"])

    if label in {"POSITIVE", "LABEL_1"}:
        return score
    if label in {"NEGATIVE", "LABEL_0"}:
        return 1.0 - score

    if label == "POSITIVE":
        return score
    if label == "NEGATIVE":
        return 1.0 - score
    if label == "NEUTRAL":
        return 0.5

    if label == "LABEL_2":
        return score
    if label == "LABEL_1":
        return 0.5
    if label == "LABEL_0":
        return 1.0 - score

    return 0.5

def classifier2_is_neutral(prefix: str) -> Tuple[bool, Optional[float], Optional[str]]:
    if sent2 is None:
        return True, None, None

    try:
        res2 = sent2(prefix)[0]
        label2 = str(res2["label"]).strip().upper()
        p2 = p_pos_from_result_any(res2)


        if label2 in {"NEUTRAL", "LABEL_1"} and ("twitter-roberta" in str(sent2_name).lower() if sent2_name else False):
            return True, p2, None


        if LOW_NEUTRAL <= p2 <= HIGH_NEUTRAL:
            return True, p2, None

        return False, p2, f"clf2_outside_band:{label2}:{res2['score']:.3f}"
    except Exception as e:
        return False, None, f"clf2_error:{e}"

# def is_neutral(
#     prefix: str,
#     low: float = LOW_NEUTRAL,
#     high: float = HIGH_NEUTRAL,
#     use_second_pass: bool = True
# ) -> Tuple[bool, float, Optional[float], Optional[str]]:
#     res1 = sent(prefix)[0]
#     p1 = p_pos_from_result_any(res1)

#     if not (low <= p1 <= high):
#         return False, p1, None, f"clf1_outside_band:{res1['label']}:{res1['score']:.3f}"

#     if use_second_pass:
#         ok2, p2, reason2 = classifier2_is_neutral(prefix)
#         if not ok2:
#             return False, p1, p2, reason2
#         return True, p1, p2, None

#     return True, p1, None, None

def collect_neutral_unique_prefixes(
    texts: List[str],
    target: int = 100,
    start_n: int = 5,
    max_n: int = 20,
    neutral_band: float = 0.05,
    max_texts_to_scan: int = 20000,
    use_second_pass: bool = True,
    return_rejections: bool = True
):
    tokenized = [tokenize(t) for t in texts[:max_texts_to_scan]]
    selected = []
    counts = {}
    seen = set()
    rejections = []

    for n in range(start_n, max_n + 1):
        if len(selected) >= target:
            counts[n] = 0
            continue

        added = 0

        for toks in tokenized:
            if len(selected) >= target:
                break
            if len(toks) < n:
                continue

            pref = " ".join(toks[:n])
            if pref in seen:
                continue

            has_block, blocked_tok = contains_blocklisted_word(pref)
            if has_block:
                if return_rejections:
                    rejections.append({
                        "n_words": n,
                        "prefix": pref,
                        "reason": "blocklist",
                        "detail": blocked_tok
                    })
                continue


            has_pat, pat_str = matches_valence_pattern(pref)
            if has_pat:
                if return_rejections:
                    rejections.append({
                        "n_words": n,
                        "prefix": pref,
                        "reason": "valence_pattern",
                        "detail": pat_str
                    })
                continue

            # ok, p1, p2, reason = is_neutral(pref, use_second_pass=use_second_pass)
            # if not ok:
              p_pos = sentiment_pos_prob(pref)
              if True:
                if return_rejections:
                    rejections.append({
                        "n_words": n,
                        "prefix": pref,
                        "reason": "classifier_reject",
                        "detail": reason
                    })
                continue

            seen.add(pref)
            selected.append((n, pref, p1, p2))
            added += 1

        counts[n] = added

    if return_rejections:
        return selected, counts, rejections
    return selected, counts

In [ ]:
# ---- Run ----
root = download_and_extract_imdb("/content/data")
texts = list(iter_review_texts(root, split="train", labels=("pos","neg")))

prefixes, counts, rejected = collect_neutral_unique_prefixes(
    texts,
    target=50000,
    start_n=5,
    max_n=20,
    neutral_band=0.05,
    max_texts_to_scan=20000,
    use_second_pass=True,
    return_rejections=True
)

print("Neutral prefixes added by n:")
for n in sorted(counts):
    if counts[n] > 0:
        print(f"  {n} words: {counts[n]}")

print(f"\nTotal collected: {len(prefixes)}")
print(f"Total rejected: {len(rejected)}\n")

for i, (n, p, p1, p2) in enumerate(prefixes, start=1):
    p2_str = "None" if p2 is None else f"{p2:.3f}"
    print(f"{i:3d}. [{n}w | p1={p1:.3f} | p2={p2_str}] {p}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Neutral prefixes added by n:
  5 words: 16647
  6 words: 17077
  7 words: 16276

Total collected: 50000
Total rejected: 6854

  1. [5w | p1=0.997 | p2=None] i think the film is
  2. [5w | p1=1.000 | p2=None] this is one of my
  3. [5w | p1=1.000 | p2=None] i'm certainly glad that a
  4. [5w | p1=0.003 | p2=None] rko studios decided to borrow
  5. [5w | p1=0.025 | p2=None] virgil manoven is an old
  6. [5w | p1=0.988 | p2=None] i thought the film could
  7. [5w | p1=1.000 | p2=None] somerset maugham's characters are brought
  8. [5w | p1=0.994 | p2=None] i was not only an
  9. [5w | p1=0.761 | p2=None] as an employee of the
 10. [5w | p1=0.002 | p2=None] i don't think a movie
 11. [5w | p1=0.923 | p2=None] i totally agree that nothing
 12. [5w | p1=1.000 | p2=None] really touching story of a
 13. [5w | p1=0.988 | p2=None] i found this movie thought
 14. [5w | p1=1.000 | p2=None] outragously entertaining period piece set
 15. [5w | p1=0.988 | p2=None] the first mystery is to
 16. [5w | p

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

df = pd.DataFrame(prefixes, columns=["n_words", "prefix", "p_pos_main", "p_pos_second"])

out_path = Path("/content/drive/MyDrive/cs175/all_prefixes.csv")
df.to_csv(out_path, index=False)

print("Saved to:", out_path)
